# Setup


In [ ]:
HADOOP_START_FROM_SCRATCH = True
DOCKER_INTERNAL_HOST = "host.docker.internal"
DOCKER_DNS = ["10.15.20.1"]

HADOOP_NAMENODE_HOSTNAME = "namenode.fbarbape.vpn.itam.mx"
HADOOP_NAMENODE_IP = "10.15.20.11"
HADOOP_NAMENODE_PORT = 8020
HADOOP_NAMENODE_WEBUI_PORT = 9870

HADOOP_RESOURCEMANAGER_HOSTNAME = "resourcemanager.fbarbape.vpn.itam.mx"
HADOOP_RESOURCEMANAGER_IP = "10.15.20.11"
HADOOP_RESOURCEMANAGER_WEBUI_PORT = 8088
HADOOP_RESOURCEMANAGER_RPC_APP_MANAGER_PORT = 8032
HADOOP_RESOURCEMANAGER_TRACKER_PORT = 8031
HADOOP_RESOURCEMANAGER_SCHEDULER_PORT = 8030
HADOOP_RESOURCEMANAGER_ADMIN_PORT = 8033

HADOOP_REPLICATION = 3
HADOOP_NUM_WORKERS = 3

HADOOP_DATANODE_IPS = ["10.15.20.11"] * 3
HADOOP_DATANODE_NAMES = [f"datanode-{i+1}" for i in range(HADOOP_NUM_WORKERS)]
HADOOP_DATANODE_HOSTNAMES = [
    f"{HADOOP_DATANODE_NAMES[i]}.fbarbape.vpn.itam.mx"
    for i in range(HADOOP_NUM_WORKERS)
]
HADOOP_DATANODE_WEBUI_PORTS = [9864 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]
HADOOP_DATANODE_TRANSFER_PORTS = [9866 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]
HADOOP_DATANODE_IPC_PORTS = [6867 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]

HADOOP_NODEMANAGER_IPS = ["10.15.20.11"] * 3
HADOOP_NODEMANAGER_NAMES = [f"nodemanager-{i+1}" for i in range(HADOOP_NUM_WORKERS)]
HADOOP_NODEMANAGER_HOSTNAMES = [
    f"{HADOOP_NODEMANAGER_NAMES[i]}.fbarbape.vpn.itam.mx"
    for i in range(HADOOP_NUM_WORKERS)
]
HADOOP_NODEMANAGER_WEBUI_PORTS = [8050 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]
HADOOP_NODEMANAGER_RPC_PORTS = [8051 + (i * 10) for i in range(HADOOP_NUM_WORKERS)]

HADOOP_WORKDIR = "/opt/hadoop/work-dir"
HADOOP_NAMENODE_NAMEDIR = "/opt/hadoop/dfs/name"
HADOOP_DATANODE_DATADIR = "/opt/hadoop/dfs/data"

HADOOP_HDFS_DATADIR = "/opt/hadoop/work-dir"

In [ ]:
import os
from pathlib import Path

LOCALHOST_WORKDIR = f"{os.path.join(os.path.relpath(Path.cwd()))}"
DOCKER_MOUNTDIR = os.path.join(LOCALHOST_WORKDIR, "mount")

Path(DOCKER_MOUNTDIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# import os
# import csv
# import random
# from faker import Faker
# from tqdm import tqdm

# fake = Faker()

# if HADOOP_START_FROM_SCRATCH:

#     def generate_data(records=100000):
#         print(f"Generating {records} records...")
#         with open(
#             os.path.join(LOCALHOST_WORKDIR, "data.csv"), "w", newline=""
#         ) as data_file:
#             writer = csv.writer(data_file)
#             writer.writerow(["ts", "id", "user", "amount", "category", "country"])
#             for _ in tqdm(range(records), desc="Progress", unit="rows"):
#                 writer.writerow(
#                     [
#                         fake.date_time_this_year().strftime("%Y-%m-%d %H:%M:%S"),
#                         fake.uuid4(),
#                         fake.name(),
#                         round(random.uniform(10.50, 10000.00), 2),
#                         fake.bs(),
#                         fake.country(),
#                     ]
#                 )

#     generate_data(records=2000000)
#     print("\nFile 'data.csv' created successfully.")

Generating 2000000 records...


Progress: 100%|██████████| 2000000/2000000 [07:41<00:00, 4330.82rows/s]


File 'data.csv' created successfully.


In [ ]:
import shutil

dataset_source_path = os.path.join(LOCALHOST_WORKDIR, "data.csv")
dataset_dest_path = os.path.join(DOCKER_MOUNTDIR, "namenode", "work-dir", "data.csv")
if HADOOP_START_FROM_SCRATCH or not os.path.exists(dataset_dest_path):
    shutil.copy(dataset_source_path, dataset_dest_path)

### Create HDFS input directory and clear previous output


In [ ]:
!docker exec namenode hdfs dfs -mkdir -p {HADOOP_HDFS_DATADIR}/input
!docker exec namenode hdfs dfs -rm -r -f {HADOOP_HDFS_DATADIR}/output
print("HDFS environment initialized.")

HDFS environment initialized.


### Upload from the container's mount point to HDFS


In [ ]:
!docker exec namenode hdfs dfs -put -f {HADOOP_WORKDIR}/data.csv {HADOOP_HDFS_DATADIR}/input/
!docker exec namenode hdfs dfs -ls {HADOOP_HDFS_DATADIR}/input

Found 1 items
-rw-r--r--   3 hadoop supergroup  244793676 2026-04-07 01:15 /opt/hadoop/work-dir/input/data.csv


### Check block locations and replication across datanodes


In [ ]:
!docker exec namenode hdfs fsck {HADOOP_HDFS_DATADIR}/input/data.csv -files -blocks -locations

FSCK started by hadoop (auth:SIMPLE) from /172.20.0.2 for path /opt/hadoop/work-dir/input/data.csv at Tue Apr 07 01:16:57 UTC 2026

/opt/hadoop/work-dir/input/data.csv 244793676 bytes, replicated: replication=3, 2 block(s):  OK
0. BP-1719492223-172.20.0.2-1775523360703:blk_1073741825_1001 len=134217728 Live_repl=3  [DatanodeInfoWithStorage[172.20.0.4:9866,DS-aa6ac4a7-923b-43ba-a279-efd6a7cd8ddc,DISK], DatanodeInfoWithStorage[172.20.0.6:9886,DS-baa04ebf-0b2b-4465-835b-a739e33f0009,DISK], DatanodeInfoWithStorage[172.20.0.5:9876,DS-533d6685-dd16-4870-b3fa-cff1e4a43a64,DISK]]
1. BP-1719492223-172.20.0.2-1775523360703:blk_1073741826_1002 len=110575948 Live_repl=3  [DatanodeInfoWithStorage[172.20.0.6:9886,DS-baa04ebf-0b2b-4465-835b-a739e33f0009,DISK], DatanodeInfoWithStorage[172.20.0.4:9866,DS-aa6ac4a7-923b-43ba-a279-efd6a7cd8ddc,DISK], DatanodeInfoWithStorage[172.20.0.5:9876,DS-533d6685-dd16-4870-b3fa-cff1e4a43a64,DISK]]


Status: HEALTHY
 Number of data-nodes:	3
 Number of racks:		1
 Total

Connecting to namenode via http://namenode.fbarbape.vpn.itam.mx:9870/fsck?ugi=hadoop&files=1&blocks=1&locations=1&path=%2Fopt%2Fhadoop%2Fwork-dir%2Finput%2Fdata.csv


### Generate mapper and reducer scripts


In [ ]:
import os

mapper_file_contents = """#!/usr/bin/env python
import sys

# Standard for Hadoop Streaming: read from STDIN
for line in sys.stdin:
    line = line.strip()
    # Split the CSV line
    parts = line.split(',')
    
    # Check if we have enough columns and skip the header
    if len(parts) >= 4 and parts[0] != "ts":
        category = parts[4]
        amount = parts[3]
        
        # Output: category [TAB] amount
        # Hadoop will sort these by the key (category) before the Reducer sees them
        print ("%s\\t%s" % (category, amount))
"""

with open(os.path.join(DOCKER_MOUNTDIR,"resourcemanager", "work-dir",'mapper.py'), 'w') as mapper_file:
    mapper_file.write(mapper_file_contents)
print("Mapper script created")


reducer_file_contents = """#!/usr/bin/env python
import sys

current_category = None
current_sum = 0.0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue
        
    try:
        category, amount = line.split('\\t')
        amount = float(amount)
    except ValueError:
        continue

    # Logic: If the category changes, print the total for the previous one
    if current_category == category:
        current_sum += amount
    else:
        if current_category:
            print ("%s\\t%.2f" % (current_category, current_sum))
        current_category = category
        current_sum = amount

# Don't forget the last category!
if current_category:
    print ("%s\\t%.2f" % (current_category, current_sum))
"""

with open(os.path.join(DOCKER_MOUNTDIR,"resourcemanager", "work-dir",'reducer.py'), 'w') as reducer_file:
    reducer_file.write(reducer_file_contents)
print("Reducer script created")

!docker exec resourcemanager ls -l {HADOOP_WORKDIR}

Mapper script created
Reducer script created
total 4
-rwxrwxrwx 1 root root 529 Apr  7 01:30 mapper.py
-rwxrwxrwx 1 root root 737 Apr  7 01:30 reducer.py


### Count directly in namenode for validation


In [ ]:
shutil.copy(
    os.path.join(DOCKER_MOUNTDIR, "resourcemanager", "work-dir", "mapper.py"),
    os.path.join(DOCKER_MOUNTDIR, "namenode", "work-dir", "mapper.py"),
)
shutil.copy(
    os.path.join(
        DOCKER_MOUNTDIR, "resourcemanager", "work-dir", "reducer.py"
    ),
    os.path.join(DOCKER_MOUNTDIR, "namenode", "work-dir", "reducer.py"),
)
!docker exec namenode bash -c "cat {HADOOP_WORKDIR}/data.csv | python {HADOOP_WORKDIR}/mapper.py | sort | python {HADOOP_WORKDIR}/reducer.py"

  File "/opt/hadoop/work-dir/mapper.py", line 17
    print "%s\t%s" % (category, amount)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
SyntaxError: Missing parentheses in call to 'print'. Did you mean print(...)?
  File "/opt/hadoop/work-dir/reducer.py", line 23
    print "%s\t%.2f" % (current_category, current_sum)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
SyntaxError: Missing parentheses in call to 'print'. Did you mean print(...)?


# Hadoop map reduce


In [ ]:
# 1. Ensure the output directory is clean
!docker exec namenode hdfs dfs -rm -r -f {HADOOP_HDFS_DATADIR}/output

# 2. Submit the job from the ResourceManager to Nodemanagers
!docker exec resourcemanager yarn jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.2.jar \
    -D mapred.reduce.tasks=2 \
    -D mapreduce.map.memory.mb=1024 \
    -D mapreduce.reduce.memory.mb=1024 \
    -files {HADOOP_WORKDIR}/mapper.py,{HADOOP_WORKDIR}/reducer.py \
    -mapper "python mapper.py" \
    -reducer "python reducer.py" \
    -input {HADOOP_HDFS_DATADIR}/input/data.csv \
    -output {HADOOP_HDFS_DATADIR}/output

# 3. Show output file
!docker exec namenode hdfs dfs -ls {HADOOP_HDFS_DATADIR}/output

JAR does not exist or is not a normal file: /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.2.jar
ls: `/opt/hadoop/work-dir/output': No such file or directory


In [ ]:
# 4. Merge and sort output
!docker exec namenode hdfs dfs -getmerge {HADOOP_HDFS_DATADIR}/output {HADOOP_WORKDIR}/output.csv
!docker exec namenode bash -c "cat {HADOOP_WORKDIR}/output.csv | sort > {HADOOP_WORKDIR}/output_sorted.csv"

getmerge: `/opt/hadoop/work-dir/output': No such file or directory
cat: /opt/hadoop/work-dir/output.csv: No such file or directory
